# LiveCodeBench prompts by difficulty

Read the existing local `test*.jsonl` files and delegate prompt validation,
deduplication and export to `PromptOperation.export_prompts()`.
No download is performed. Missing data is reported without creating outputs.


## 1. Repository setup

In [1]:
# Standard library: resolve the repository before importing project modules.
import json
import sys
from pathlib import Path

working_directory = Path.cwd().resolve()
for directory in (working_directory, *working_directory.parents):
    if (directory / "modeling/src/data/export_prompts.py").is_file():
        PROJECT_ROOT = directory
        break
else:
    raise FileNotFoundError("Open this notebook inside prompt-difficulty-predictor")

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

MODELING_DIR = PROJECT_ROOT / "modeling"
DATA_DIR = MODELING_DIR / "data"


## 2. Imports and configuration

In [2]:
from collections import defaultdict

from modeling.src.data.export_prompts import PromptOperation

RAW_DIR = DATA_DIR / "raw/livecodebench"
OUTPUT_DIR = DATA_DIR / "processed/por_nivel_livecodebench"
prompt_operation = PromptOperation(data_dir=DATA_DIR)
source_files = sorted(RAW_DIR.glob("test*.jsonl"))


/Users/emiliogarcialopez/projectGitHub/deep_learning_Project/prompt-difficulty-predictor/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 3. Group local prompts

Keep the existing easy, medium and hard groups. Unknown difficulty labels are counted separately.

In [3]:
prompts_by_difficulty = defaultdict(list)
skipped_unknown_difficulty = 0

for path in source_files:
    with path.open(encoding="utf-8") as stream:
        for line in stream:
            if not line.strip():
                continue
            record = json.loads(line)
            difficulty = str(record.get("difficulty", "")).lower()
            if difficulty not in {"easy", "medium", "hard"}:
                skipped_unknown_difficulty += 1
                continue
            prompts_by_difficulty[difficulty].append(record.get("question_content"))

if not source_files:
    print(f"LiveCodeBench data is not available: {RAW_DIR}")
else:
    print(f"Source files: {len(source_files)}")
    print(f"Skipped unknown difficulties: {skipped_unknown_difficulty}")


LiveCodeBench data is not available: /Users/emiliogarcialopez/projectGitHub/deep_learning_Project/prompt-difficulty-predictor/modeling/data/raw/livecodebench


## 4. Export through the existing prompt operation

In [4]:
for difficulty, prompts in sorted(prompts_by_difficulty.items()):
    summary = prompt_operation.export_prompts(prompts, OUTPUT_DIR / f"{difficulty}.jsonl")
    print(difficulty, summary)
